# Portfolio Financing System Prototype (V3)
**Author**: Chris Hsieh (chrishsh@gmail.com)

This notebook is the prototype implementation of the **Automated Portfolio Financing System**. 
It perfectly matches the logic of the full-blown Python suite (`portfolio_financing/` package) but is presented in an interactive cell-by-cell format.

In this version, **each module is split into its own cell**, followed by an execution block to clearly display its output.

### Modules Included:
1. **Inventory Manager**: Seeds multi-strat positions.
2. **Internalization Engine**: Nets gross exposures.
3. **Compliance Engine**: US Reg T/SHO and APAC SBL checks.
4. **Locate Engine**: PB simulator for borrow rates.
5. **Collateral Optimizer**: MILP solver for optimal pledging.
6. **Manual Trade Entry**: Blotter for Repo, TRS, Sell/Buyback, etc.
7. **P&L Engine**: Actual/360 daily accruals.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import linprog
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ==========================================
# Module 1: Inventory & Market Data
# ==========================================
class InventoryManager:
    def __init__(self, tickers, strategies):
        self.tickers = tickers
        self.strategies = strategies
        self.prices = {}
        
    def fetch_market_data(self):
        for t in self.tickers:
            try:
                dat = yf.Ticker(t).history(period='1d')
                self.prices[t] = dat['Close'].iloc[-1] if not dat.empty else 100.0
            except:
                self.prices[t] = 100.0
                
    def generate_mock_inventory(self):
        records = []
        np.random.seed(42)
        for strat in self.strategies:
            for t in self.tickers:
                qty = np.random.randint(-10000, 10000)
                if qty != 0:
                    records.append({
                        'Strategy': strat, 'Ticker': t,
                        'AssetClass': 'FixedIncome' if t in ['AGG', 'TLT'] else 'Equity',
                        'Jurisdiction': 'APAC' if '.HK' in t or '.T' in t else 'US',
                        'Quantity': qty, 'MtM_Price': self.prices.get(t, 100.0),
                        'Notional': qty * self.prices.get(t, 100.0)
                    })
        return pd.DataFrame(records)

In [ ]:
# Execution: Inventory & Market Data
tickers = ['AAPL', 'MSFT', 'JPM', 'AGG', 'TLT', '0700.HK']
strats = ['StatArb', 'VolArb', 'Macro']

mgr = InventoryManager(tickers, strats)
mgr.fetch_market_data()
ledger = mgr.generate_mock_inventory()

print("Module 1 Output: Firm Master Ledger (Gross Inventory)")
display(ledger.head(10))

In [ ]:
# ==========================================
# Module 2: Internalization Engine
# ==========================================
class InternalizationEngine:
    def __init__(self, ledger):
        self.ledger = ledger
        
    def calculate_net_exposure(self):
        net_pos = self.ledger.groupby(['Ticker', 'AssetClass', 'Jurisdiction']).agg({'Quantity': 'sum', 'Notional': 'sum'}).reset_index()
        gross = self.ledger['Quantity'].abs().sum()
        net = net_pos['Quantity'].abs().sum()
        ratio = 1 - (net / gross) if gross > 0 else 0
        return net_pos, ratio

In [ ]:
# Execution: Internalization Engine
int_eng = InternalizationEngine(ledger)
net_exp, ratio = int_eng.calculate_net_exposure()

print(f"Internalization Ratio: {ratio*100:.2f}%")
print("Module 2 Output: Net External Exposure")
display(net_exp)

In [ ]:
# ==========================================
# Module 3 & 4: Locates & Compliance
# ==========================================
class SecurityLocateEngine:
    def __init__(self, tickers):
        self.rates = {t: np.random.uniform(0.0025, 0.01) for t in tickers}
        if 'AAPL' in self.rates: self.rates['AAPL'] = 0.065 # HTB
    def get_locate(self, ticker):
        return {'Locate_ID': np.random.randint(100000, 999999), 'Borrow_Fee_Rate': self.rates.get(ticker, 0.01), 'Secured': True}

class RegulatoryComplianceEngine:
    def __init__(self, locate_engine):
        self.locate_engine = locate_engine
    def validate_exposure(self, net_exposure_df):
        records = []
        for _, row in net_exposure_df.iterrows():
            record = row.to_dict()
            record['Compliance_Status'] = 'Passed'
            if row['Quantity'] < 0:
                if row['Jurisdiction'] == 'US':
                    loc = self.locate_engine.get_locate(row['Ticker'])
                    record['Locate_ID'] = loc['Locate_ID']
                    record['Borrow_Fee_Rate'] = loc['Borrow_Fee_Rate']
                elif row['Jurisdiction'] == 'APAC':
                    if '0700.HK' in row['Ticker']: # Mock failure
                        record['Compliance_Status'] = 'REJECTED: APAC Naked Short Ban'
                        records.append(record)
                        continue
                    record['Borrow_Fee_Rate'] = 0.02
            records.append(record)
        return pd.DataFrame(records)

In [ ]:
# Execution: Locates & Compliance
loc_eng = SecurityLocateEngine(tickers)
comp_eng = RegulatoryComplianceEngine(loc_eng)
valid = comp_eng.validate_exposure(net_exp)

print("Module 3/4 Output: Compliance Validated Exposures (Note 0700.HK rejection logic if short)")
display(valid)

In [ ]:
# ==========================================
# Module 5: Collateral Optimizer (MILP)
# ==========================================
class CollateralOptimizer:
    def __init__(self, inventory, margin_reqs):
        self.inventory = inventory
        self.margin_reqs = margin_reqs
        self.cost = {t: 0.05 if t in ['AGG', 'TLT'] else 0.01 for t in inventory.keys()}
        self.haircuts = {t: 0.02 if t in ['AGG', 'TLT'] else 0.10 for t in inventory.keys()}
    def optimize(self):
        tickers = list(self.inventory.keys())
        reqs = list(self.margin_reqs.keys())
        if not tickers or not reqs: return pd.DataFrame(), False
        c = np.zeros(len(tickers) * len(reqs))
        for i, t in enumerate(tickers):
            for j in range(len(reqs)): c[i * len(reqs) + j] = self.cost[t]
        A_ub, b_ub = np.zeros((len(tickers), len(tickers) * len(reqs))), np.zeros(len(tickers))
        for i in range(len(tickers)):
            for j in range(len(reqs)): A_ub[i, i * len(reqs) + j] = 1
            b_ub[i] = self.inventory[tickers[i]]
        A_req, b_req = np.zeros((len(reqs), len(tickers) * len(reqs))), np.zeros(len(reqs))
        for j in range(len(reqs)):
            for i in range(len(tickers)): A_req[j, i * len(reqs) + j] = -(1 - self.haircuts[tickers[i]])
            b_req[j] = -self.margin_reqs[reqs[j]]
        A_ub, b_ub = np.vstack((A_ub, A_req)), np.concatenate((b_ub, b_req))
        res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=(0, None), method='highs')
        allocs = []
        if res.success:
            x = res.x.reshape((len(tickers), len(reqs)))
            for i, t in enumerate(tickers):
                for j, r in enumerate(reqs):
                    if x[i,j] > 0.01: allocs.append({'Ticker': t, 'CP': r, 'Notional': x[i,j]})
        return pd.DataFrame(allocs), res.success

In [ ]:
# Execution: Collateral Optimizer
longs = valid[(valid['Quantity'] > 0) & (valid['Compliance_Status'] == 'Passed')]
long_inv = longs.groupby('Ticker')['Notional'].sum().to_dict()
margin_reqs = {'PrimeBroker_A': 1500000, 'ClearingHouse_B': 500000}

print("Available Long Inventory:", long_inv)
print("Margin Requirements:", margin_reqs)

opt = CollateralOptimizer(long_inv, margin_reqs)
alloc, success = opt.optimize()

print(f"\nOptimization Success: {success}")
if success:
    display(alloc)

In [ ]:
# ==========================================
# Module 6: Manual Trade Entry
# ==========================================
class ManualTradeEntry:
    def __init__(self):
        self.trades = []
    
    def _add_trade(self, trade_dict):
        trade_dict['Trade_ID'] = f"M-{len(self.trades)+1}"
        trade_dict['Timestamp'] = datetime.now().isoformat()
        self.trades.append(trade_dict)
        
    def book_repo(self, collateral_ticker, cash_principal, rate, direction="Repo"):
        self._add_trade({'Type': direction, 'Ticker': collateral_ticker, 'Notional': cash_principal, 'Rate': rate})
        
    def book_trs(self, ticker, quantity, price, direction, spread):
        self._add_trade({'Type': 'TRS', 'Ticker': ticker, 'Quantity': quantity, 'Spot_Price': price, 'Notional': quantity * price, 'Direction': direction, 'Rate': spread})
        
    def book_sec_borrow_loan(self, ticker, quantity, price, rate, direction):
        self._add_trade({'Type': f'Securities {direction}', 'Ticker': ticker, 'Quantity': quantity, 'Spot_Price': price, 'Notional': quantity * price, 'Rate': rate})
        
    def book_cash_financing(self, principal, rate, direction):
        self._add_trade({'Type': f'Cash {direction}', 'Ticker': 'USD', 'Notional': principal, 'Rate': rate})
        
    def book_sell_buyback(self, ticker, notional, spot_price, forward_price):
        self._add_trade({'Type': 'Sell Buyback', 'Ticker': ticker, 'Notional': notional, 'Spot_Price': spot_price, 'Forward_Price': forward_price, 'Rate': 0.0})
        
    def get_blotter(self):
        if not self.trades: return pd.DataFrame(columns=['Trade_ID', 'Type', 'Ticker', 'Notional', 'Rate', 'Timestamp'])
        return pd.DataFrame(self.trades)

In [ ]:
# Execution: Manual Trade Entry Demo
manual = ManualTradeEntry()
manual.book_repo('AGG', 5000000, 0.0525)
manual.book_trs('AAPL', 10000, 150.0, 'Receive', 0.005)
manual.book_sell_buyback('TLT', 1000000, 98.50, 99.00)

print("Module 6 Output: Manual Trade Blotter")
display(manual.get_blotter())

In [ ]:
# ==========================================
# Module 7: P&L Engine
# ==========================================
class PnLEngine:
    def __init__(self, prices, benchmark=0.05):
        self.prices = prices
        self.benchmark = benchmark
        
    def calc_auto(self, df):
        if df.empty: return pd.DataFrame(), 0
        recs = []
        tot = 0
        for _, r in df.iterrows():
            if r['Compliance_Status'] != 'Passed': continue
            notional = abs(r['Quantity'] * self.prices.get(r['Ticker'], 100))
            if r['Quantity'] < 0:
                c = -(notional * r.get('Borrow_Fee_Rate', 0.01)) / 360
                recs.append({'Ticker': r['Ticker'], 'Type': 'Auto Short Borrow', 'PnL': c}); tot += c
            else:
                c = -(notional * self.benchmark) / 360
                recs.append({'Ticker': r['Ticker'], 'Type': 'Auto Long Financing', 'PnL': c}); tot += c
        return pd.DataFrame(recs), tot

    def calc_manual(self, blotter, price_change=0.0):
        if blotter.empty: return pd.DataFrame(), 0
        recs = []
        tot = 0
        for _, r in blotter.iterrows():
            typ = r['Type']
            pnl = 0.0
            if typ == 'Repo': pnl = -(r['Notional'] * r['Rate']) / 360
            elif typ == 'Reverse Repo': pnl = (r['Notional'] * r['Rate']) / 360
            elif typ == 'TRS':
                fin = -(r['Notional'] * (self.benchmark + r['Rate'])) / 360
                perf = r.get('Quantity', 0) * price_change
                if r.get('Direction') == 'Pay': perf = -perf
                pnl = fin + perf
            elif typ == 'Securities Borrow': pnl = -(r['Notional'] * r['Rate']) / 360
            elif typ == 'Securities Loan': pnl = (r['Notional'] * r['Rate']) / 360
            elif typ == 'Cash Borrow': pnl = -(r['Notional'] * r['Rate']) / 360
            elif typ == 'Cash Lend': pnl = (r['Notional'] * r['Rate']) / 360
            elif typ == 'Sell Buyback':
                implied_rate = (r['Forward_Price'] - r['Spot_Price']) / r['Spot_Price']
                pnl = -(r['Notional'] * implied_rate) / 360
            tot += pnl
            recs.append({'Ticker': r['Ticker'], 'Type': typ, 'PnL': pnl})
        return pd.DataFrame(recs), tot

In [ ]:
# Execution: P&L Engine
pnl = PnLEngine(mgr.prices)
auto_df, auto_tot = pnl.calc_auto(valid)
man_df, man_tot = pnl.calc_manual(manual.get_blotter(), price_change=5.0)

print(f"Total Daily Financing P&L: ${auto_tot + man_tot:,.2f}")
print("\nAutomated P&L:")
display(auto_df)
print("\nManual P&L:")
display(man_df)